In [1]:
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

In [2]:
import pathlib
import torch
from tqdm import tqdm
from torchmetrics.regression import (
    R2Score,
    MeanSquaredError,
    MeanAbsoluteError,
    PearsonCorrCoef,
)

from egnn.qm9.models import EGNN
from egnn.qm9 import utils as qm9_utils
from src.dataset import EGNNCP3DDataModule

torch.set_float32_matmul_precision("high")

/home/pham/miniforge3/envs/cp3d/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class ARGS:
    def __init__(self, configs):
        for key, value in configs.items():
            setattr(self, key, value)


configs = {
    "data_dir": pathlib.Path("data/egnn/Hexene"),
    "seed": 43,
    "amp": True,
    "log_dir": pathlib.Path("./results/egnn/cp3d"),
    "epochs": 15,
    "eval_interval": 2,
    "save_ckpt_path": pathlib.Path("./results/egnn/cp3d/new_env_test_egnn.pth"),
    "learning_rate": 0.00021,
    "min_learning_rate": 1e-7,
    "batch_size": 32,
    "nf": 128,
    "n_layers": 7,
    "attention": 1,
    "node_attr": 0,
    "property": "pampa",
    "charge_power": 2,
    "split": 0,
}

args = ARGS(configs)

In [4]:
model = EGNN(
    in_node_nf=12,
    in_edge_nf=0,
    hidden_nf=args.nf,
    n_layers=args.n_layers,
    coords_weight=1.0,
    attention=args.attention,
    node_attr=args.node_attr,
)

In [5]:
loss_fn = torch.nn.L1Loss()
datamodule = EGNNCP3DDataModule(
    args.data_dir, args.split, batch_size=args.batch_size, num_workers=4
)
train_dataloader = datamodule.train_dataloader()
val_dataloader = datamodule.val_dataloader()

In [6]:
values = train_dataloader.dataset.data[args.property]
mean = torch.mean(values)
mad = torch.mean(torch.abs(values - mean))

In [7]:
device = torch.cuda.current_device()
model.to(device=device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=args.learning_rate,
    fused=True,
)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, args.epochs)

mae = MeanAbsoluteError()
rmse = MeanSquaredError(squared=False)
r2 = R2Score()
pearson = PearsonCorrCoef()

for epoch in range(args.epochs):
    loss_acc = torch.zeros((1,), device=device)
    model.train()
    for i, batch in tqdm(
        enumerate(train_dataloader),
        total=len(train_dataloader),
        unit="batch",
        desc=f"Epoch {epoch + 1}",
    ):
        batch_size, n_nodes, _ = batch["positions"].size()
        atom_positions = (
            batch["positions"]
            .view(batch_size * n_nodes, -1)
            .to(device, torch.float32, non_blocking=True)
        )
        atom_mask = (
            batch["atom_mask"]
            .view(batch_size * n_nodes, -1)
            .to(device, torch.float32, non_blocking=True)
        )
        edge_mask = batch["edge_mask"].to(device, torch.float32, non_blocking=True)
        one_hot = batch["one_hot"].to(device, torch.float32, non_blocking=True)
        charges = batch["charges"].to(device, torch.float32, non_blocking=True)
        nodes = qm9_utils.preprocess_input(
            one_hot, charges, args.charge_power, datamodule.max_charge, device
        )

        nodes = nodes.view(batch_size * n_nodes, -1)
        edges = qm9_utils.get_adj_matrix(n_nodes, batch_size, device)
        target = batch[args.property].to(device, torch.float32, non_blocking=True)

        pred = model(
            h0=nodes,
            x=atom_positions,
            edges=edges,
            edge_attr=None,
            node_mask=atom_mask,
            edge_mask=edge_mask,
            n_nodes=n_nodes,
        )
        loss = loss_fn(pred.flatten(), (target.flatten() - mean) / mad)
        loss_acc += loss.detach()
        loss.backward()
        optimizer.step()
        model.zero_grad(set_to_none=True)
        # lr_scheduler.step()
    print(f"Epoch {epoch + 1}")
    print(f"Train Loss {loss_acc.item() / len(train_dataloader):.4f}")

    model.eval()
    with torch.inference_mode():
        loss_acc = torch.zeros((1,), device="cpu")
        for i, batch in tqdm(
            enumerate(val_dataloader),
            total=len(val_dataloader),
            unit="batch",
            desc=f"Epoch {epoch + 1}",
            leave=False,
        ):
            batch_size, n_nodes, _ = batch["positions"].size()
            atom_positions = (
                batch["positions"]
                .view(batch_size * n_nodes, -1)
                .to(device, torch.float32, non_blocking=True)
            )
            atom_mask = (
                batch["atom_mask"]
                .view(batch_size * n_nodes, -1)
                .to(device, torch.float32, non_blocking=True)
            )
            edge_mask = batch["edge_mask"].to(device, torch.float32, non_blocking=True)
            one_hot = batch["one_hot"].to(device, torch.float32, non_blocking=True)
            charges = batch["charges"].to(device, torch.float32, non_blocking=True)
            nodes = qm9_utils.preprocess_input(
                one_hot, charges, args.charge_power, datamodule.max_charge, device
            )

            nodes = nodes.view(batch_size * n_nodes, -1)
            edges = qm9_utils.get_adj_matrix(n_nodes, batch_size, device)
            target = batch[args.property].to(device, torch.float32, non_blocking=True)

            pred = model(
                h0=nodes,
                x=atom_positions,
                edges=edges,
                edge_attr=None,
                node_mask=atom_mask,
                edge_mask=edge_mask,
                n_nodes=n_nodes,
            )
            pred = mad * pred + mean
            pred_flat, target_flat = (
                pred.detach().view(-1).float().cpu(),
                target.detach().view(-1).float().cpu(),
            )

            val_loss = loss_fn(pred_flat, target_flat)
            loss_acc += val_loss.detach()

            mae(pred_flat, target_flat)
            rmse(pred_flat, target_flat)
            r2(pred_flat, target_flat)
            pearson(pred_flat, target_flat)

        print(f"Val Loss {loss_acc.item() / len(val_dataloader):.4f}")
        print(
            f"""
            validation MAE: {float(mae.compute()):.4f}
            validation RMSE: {float(rmse.compute()):.4f}
            validation R²: {float(r2.compute()):.4f}
            validation Pearson r: {float(pearson.compute()):.4f}
            """
        )

        mae.reset()
        rmse.reset()
        r2.reset()
        pearson.reset()

Epoch 1: 100%|██████████| 129/129 [00:27<00:00,  4.64batch/s]


Epoch 1
Train Loss 169.4215


Val Loss 1.2543

            validation MAE: 1.2177
            validation RMSE: 1.3999
            validation R²: -13.0239
            validation Pearson r: 0.0814
            


Epoch 2: 100%|██████████| 129/129 [00:27<00:00,  4.73batch/s]


Epoch 2
Train Loss 2.0267


...

Epoch 14
Train Loss 1.1385


Val Loss 0.2969

            validation MAE: 0.2873
            validation RMSE: 0.3754
            validation R²: -0.0083
            validation Pearson r: 0.2288
            


Epoch 15: 100%|██████████| 129/129 [00:27<00:00,  4.72batch/s]


Epoch 15
Train Loss 1.1898


Val Loss 0.4105

            validation MAE: 0.3993
            validation RMSE: 0.4738
            validation R²: -0.6063
            validation Pearson r: 0.1422
            
